# EDA - Face Mask Dataset

Project 4 - Face Mask Detection (Advanced ML team project).

Goal of this notebook: just look at the data before we train anything. We need to know:
- how many images per class
- if there is class imbalance
- image sizes / formats
- some sample images so we know what we're dealing with

Dataset: ashishjangra27/face-mask-12k-images-dataset (Train / Validation / Test splits already done by the dataset author).

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

# kaggle path on colab, change if running locally
import os
BASE = os.environ.get("DATASET_PATH", "../data")
TRAIN_DIR = os.path.join(BASE, "Train")
VAL_DIR   = os.path.join(BASE, "Validation")
TEST_DIR  = os.path.join(BASE, "Test")

# fix seed so the random samples are the same when we re-run
random.seed(42)
np.random.seed(42)

sns.set_style("whitegrid")
print("ok")

## 1) How many images do we have per split / per class?

First sanity check. The dataset says 12k images in total but let's count ourselves.

In [ ]:
def count_images(folder):
    counts = {}
    if not os.path.isdir(folder):
        return counts
    for cls in sorted(os.listdir(folder)):
        cls_path = os.path.join(folder, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([f for f in os.listdir(cls_path)
                               if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR)
test_counts  = count_images(TEST_DIR)

df_counts = pd.DataFrame({
    "Train": train_counts,
    "Validation": val_counts,
    "Test": test_counts
}).fillna(0).astype(int)
df_counts.loc["TOTAL"] = df_counts.sum()
df_counts

**Insight 1:** Train ~10k, Validation only ~800, Test ~990. So validation is small (around 8% of training). We'll need to be careful — a small val set means noisy validation accuracy, and we might want to also keep some test images aside for the final report.

## 2) Class balance plot (train set)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (split_name, counts) in zip(axes, [("Train", train_counts), ("Validation", val_counts), ("Test", test_counts)]):
    if not counts:
        continue
    ax.bar(list(counts.keys()), list(counts.values()), color=["#4C9F70", "#D9534F"])
    ax.set_title(f"{split_name} - class distribution")
    ax.set_ylabel("# images")
    for i, v in enumerate(counts.values()):
        ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

**Insight 2:** The train set is almost perfectly balanced (around 5000 each). That's good — no need for WeightedRandomSampler or class weights for this dataset. (Different from the X-ray dataset in project 2 where we'd actually need it.)

## 3) Look at sample images per class

We need to actually SEE the images, not just read counts. Sometimes datasets have weird stuff (watermarks, labels printed on the image, etc).

In [ ]:
def show_grid(folder, n=8, title=""):
    files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    picks = random.sample(files, min(n, len(files)))
    fig, axs = plt.subplots(2, 4, figsize=(12, 6))
    fig.suptitle(title, fontsize=14)
    for ax, f in zip(axs.flatten(), picks):
        img = Image.open(os.path.join(folder, f))
        ax.imshow(img)
        ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

for cls in train_counts.keys():
    show_grid(os.path.join(TRAIN_DIR, cls), n=8, title=f"Samples - {cls}")

**Insight 3:** The faces are very different — different ages, lighting, mask types (cloth, surgical, n95). Some images are basically just a face from far away. Image dimensions are not consistent (some 200x200, some bigger). So `transforms.Resize((224,224))` is mandatory.

## 4) Image size distribution

Reading the size of every image. A bit slow but useful — we want to know if we have weird tiny or weird huge images that we should filter out.

In [ ]:
def collect_sizes(folder, sample=1000):
    rows = []
    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)
        if not os.path.isdir(cls_path):
            continue
        files = [f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        # don't read all 5k each, take a sample for speed
        files = random.sample(files, min(sample, len(files)))
        for f in files:
            try:
                with Image.open(os.path.join(cls_path, f)) as im:
                    rows.append({"class": cls, "width": im.size[0], "height": im.size[1], "mode": im.mode})
            except Exception as e:
                pass # skip corrupted
    return pd.DataFrame(rows)

sizes_df = collect_sizes(TRAIN_DIR, sample=800)
sizes_df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(sizes_df["width"], bins=30, color="#4C9F70", alpha=0.7, label="width")
axes[0].hist(sizes_df["height"], bins=30, color="#D9534F", alpha=0.7, label="height")
axes[0].legend(); axes[0].set_title("Image dimensions distribution")
axes[0].set_xlabel("pixels")

axes[1].scatter(sizes_df["width"], sizes_df["height"], alpha=0.3, s=8)
axes[1].set_xlabel("width"); axes[1].set_ylabel("height")
axes[1].set_title("Width vs Height scatter")
plt.tight_layout(); plt.show()

**Insight 4:** Most images are around 200x200 to 350x350. Almost square. So resize to 224x224 (which is what MobileNetV2 wants anyway) is not gonna distort the faces much.

## 5) Color mode check

Are all images RGB? Sometimes there are grayscale or RGBA images and they break the dataloader.

In [ ]:
mode_counts = sizes_df["mode"].value_counts()
print(mode_counts)
mode_counts.plot(kind="bar", color="#5B9BD5")
plt.title("Image color mode"); plt.xticks(rotation=0); plt.show()

**Insight 5:** If we see anything other than RGB we'll just convert to RGB inside the transform pipeline. (Already handled by `.convert('RGB')` in the API too.)

## 6) Mean pixel intensity per class

Sometimes one class is just on average darker/brighter than the other — the model could learn from background brightness instead of the actual mask. Want to check that.

In [ ]:
def mean_intensity(folder, n=200):
    files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    files = random.sample(files, min(n, len(files)))
    means = []
    for f in files:
        try:
            arr = np.array(Image.open(os.path.join(folder, f)).convert("RGB"))
            means.append(arr.mean())
        except:
            pass
    return means

intensities = {cls: mean_intensity(os.path.join(TRAIN_DIR, cls), 250) for cls in train_counts.keys()}

plt.figure(figsize=(9,4))
for cls, vals in intensities.items():
    plt.hist(vals, bins=25, alpha=0.55, label=cls)
plt.legend(); plt.title("Mean pixel intensity per class")
plt.xlabel("mean RGB value"); plt.ylabel("count")
plt.show()

**Insight 6:** The two distributions overlap a lot — model can't cheat by just looking at brightness. Good.

## 7) Aspect ratio histogram

In [ ]:
sizes_df["aspect"] = sizes_df["width"] / sizes_df["height"]
plt.figure(figsize=(9,4))
plt.hist(sizes_df["aspect"], bins=40, color="#9467BD")
plt.axvline(1.0, color="red", linestyle="--", label="square (1:1)")
plt.title("Aspect ratio (w/h) distribution")
plt.legend(); plt.show()

**Insight 7:** Most images are close to 1.0 (square), some are slightly portrait (taller). Resize((224,224)) won't be a problem.

## 8) Side-by-side WithMask vs WithoutMask

One last visual — same person with vs without isn't possible here (it's not paired) but seeing them side by side helps.

In [ ]:
classes = list(train_counts.keys())
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for row, cls in enumerate(classes):
    folder = os.path.join(TRAIN_DIR, cls)
    files = random.sample(os.listdir(folder), 5)
    for col, f in enumerate(files):
        img = Image.open(os.path.join(folder, f))
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis("off")
plt.tight_layout(); plt.show()

## 9) Augmentation preview - what does the training pipeline actually see?

Before training, we want to confirm the augmentations look sane.
If they're too aggressive (e.g. rotated 180 degrees) the model will learn garbage.

In [ ]:
from torchvision import transforms

aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
])

# pick one sample image and apply augmentation 8 times
sample_cls = list(train_counts.keys())[0]
sample_files = os.listdir(os.path.join(TRAIN_DIR, sample_cls))
img = Image.open(os.path.join(TRAIN_DIR, sample_cls, sample_files[0])).convert("RGB")

fig, axs = plt.subplots(2, 4, figsize=(13, 6))
fig.suptitle("Augmentation preview - same image, 8 random augmentations", fontsize=13)
for ax in axs.flatten():
    ax.imshow(aug(img))
    ax.axis("off")
plt.tight_layout(); plt.show()

**Insight 8:** Augmentations look reasonable. Faces stay recognizable, lighting changes are realistic, no extreme distortions. Safe to use.

## Summary of EDA findings

1. Total ~12k images, train ~10k, val ~800, test ~990.
2. Train set is balanced between WithMask / WithoutMask. No class weighting needed.
3. Validation is small (800) — val accuracy will be a bit noisy.
4. Image sizes vary but most are ~200-350 px and roughly square. Resize((224,224)) is fine.
5. Both classes have similar mean pixel intensity → model has to actually look at the face.
6. Augmentation plan: HorizontalFlip (faces are symmetric), small Rotation (±15°), ColorJitter (indoor/outdoor lighting). NO vertical flip — upside-down faces don't help.
7. We'll use MobileNetV2 (pretrained on ImageNet), freeze backbone, train classifier head only — small dataset + 2 classes, no need for a heavy model.
8. Target accuracy from the project guidelines: >90%. Should be reachable easily.